In [ ]:
import serial
import time
import math
import pyvisa
import csv
from datetime import datetime
from simple_pid import PID

# ================= CONFIGURATION =================
ARDUINO_PORT = 'COM9' 
KEITHLEY_VISA = 'USB0::0x05E6::0x2450::04542648::INSTR' 

# Thermistor Constants
V_IN = 5.135
R_FIXED = 10000.0  
R0 = 10000.0       
T0 = 298.15        
BETA = 3435.0

# PID & Heater Settings
TARGET_TEMP_C = 28.0   
MAX_HEATER_VOLTS = 7.0 
CURRENT_LIMIT = 0.4    

pid = PID(1.2, 0.08, 0.03, setpoint=TARGET_TEMP_C)
pid.output_limits = (0, MAX_HEATER_VOLTS)

log_filename = f"heater_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

# Sampling Interval in seconds
SAMPLE_INTERVAL = 1.0 
# =================================================

def get_temp(v_measured):
    if v_measured <= 0.1 or v_measured >= 4.9:
        return None 
    r_ntc = R_FIXED * ((V_IN / v_measured) - 1)
    try:
        steinhart = math.log(r_ntc / R0) / BETA
        steinhart += 1.0 / T0
        return (1.0 / steinhart) - 273.15
    except:
        return None

# ================= INITIALIZATION =================
print(f"Initializing Logging to: {log_filename}")
rm = pyvisa.ResourceManager()
smu = rm.open_resource(KEITHLEY_VISA)
smu.read_termination = '\n'
smu.write_termination = '\n'
smu.timeout = 10000 

ser = serial.Serial(ARDUINO_PORT, 115200, timeout=1)
time.sleep(2)

smu.write("*RST")
smu.write(":SOUR:FUNC VOLT")
smu.write(f":SOUR:VOLT:ILIM {CURRENT_LIMIT}")
smu.write(":SOUR:VOLT 0")
smu.write(":OUTP ON")

# ================= MAIN LOOP =================
with open(log_filename, mode='w', newline='') as csvfile:
    log_writer = csv.writer(csvfile)
    log_writer.writerow(['Elapsed_Time_S', 'Temp_C', 'Setpoint_C', 'PID_Voltage_V', 'SMU_Actual_V', 'SMU_Actual_I'])

    print(f"PID Loop Active. Interval: {SAMPLE_INTERVAL}s. Recording...")
    
    start_time = time.time()
    last_update_time = 0  # To track the 1-second intervals

    try:
        while True:
            current_time = time.time() - start_time
            
            # Logic Gate: Only run if 1 second has passed since the last update
            if current_time - last_update_time >= SAMPLE_INTERVAL:
                
                # Clear the serial buffer so we get the 'freshest' reading
                ser.reset_input_buffer()
                
                # Wait for the next fresh line from Arduino
                line = ser.readline().decode('utf-8').strip()
                
                try:
                    v_read = float(line)
                    temp = get_temp(v_read)
                    
                    if temp is not None:
                        # 1. PID Logic
                        new_v = pid(temp)
                        smu.write(f":SOUR:VOLT {new_v:.4f}")
                        
                        # 2. Query Keithley (Wrapped in try/except for stability)
                        try:
                            smu_data = smu.query(":READ? 'defbuffer1', SOUR, READ")
                            actual_v, actual_i = smu_data.split(',')
                        except:
                            actual_v, actual_i = "Err", "Err"
                        
                        # 3. Save to CSV
                        log_writer.writerow([f"{current_time:.2f}", f"{temp:.3f}", TARGET_TEMP_C, 
                                            f"{new_v:.3f}", actual_v.strip(), actual_i.strip()])
                        csvfile.flush() 
                        
                        # 4. Console Output
                        print(f"Time: {current_time:.2f} s | Temp: {temp:.2f} C | PID: {new_v:.2f} V")
                        
                        # Update the timestamp for the next 1-second gate
                        last_update_time = current_time
                        
                except ValueError:
                    continue

    except KeyboardInterrupt:
        print("\nShutdown signal received...")
    finally:
        smu.write(":SOUR:VOLT 0")
        smu.write(":OUTP OFF")
        smu.close()
        ser.close()
        print(f"Cleanup complete. Data saved to {log_filename}")